In [1]:
#### Li Extraction Efficiency Prediction Model ####
import os
import time
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold, KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from rdkit import Chem
from rdkit.Chem import GraphDescriptors, MolSurf, Descriptors, Lipinski, rdMolDescriptors, EState
from rdkit.Chem.EState import EState_VSA
from rdkit.Chem.rdchem import Mol
from sklearn.linear_model import ElasticNet
from sklearn.feature_selection import mutual_info_regression
from sklearn.tree import DecisionTreeRegressor
import joblib
from scipy.stats import pearsonr
import seaborn as sns

# Set global font settings for matplotlib
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False

def molecular_descriptors(mol: Mol) -> dict:
    descriptors = {}

    # Original descriptors
    descriptors['Kappa1'] = GraphDescriptors.Kappa1(mol)
    descriptors['Kappa2'] = GraphDescriptors.Kappa2(mol)
    descriptors['Kappa3'] = GraphDescriptors.Kappa3(mol)
    descriptors['BertzCT'] = GraphDescriptors.BertzCT(mol)

    # PEOE_VSA descriptors
    for i in range(1, 15):
        descriptors[f'PEOE_VSA{i}'] = getattr(MolSurf, f'PEOE_VSA{i}')(mol)

    # Other descriptors
    descriptors['heavy_atom_count'] = Descriptors.HeavyAtomCount(mol)
    descriptors['rotatable_bonds'] = Descriptors.NumRotatableBonds(mol)
    descriptors['h_acceptors'] = Descriptors.NumHAcceptors(mol)
    descriptors['heteroatom_count'] = Descriptors.NumHeteroatoms(mol)
    descriptors['tpsa'] = Descriptors.TPSA(mol)
    descriptors['FpDensity'] = Descriptors.FpDensityMorgan1(mol)
    descriptors['FractionCSP3'] = Lipinski.FractionCSP3(mol)

    # Chi descriptors
    for i in range(5):
        descriptors[f'Chi{i}n'] = getattr(rdMolDescriptors, f'CalcChi{i}n')(mol)
        descriptors[f'Chi{i}v'] = getattr(rdMolDescriptors, f'CalcChi{i}v')(mol)

    # Crippen descriptors
    logp, mr = rdMolDescriptors.CalcCrippenDescriptors(mol)
    descriptors['CrippenLogP'] = logp
    descriptors['CrippenMR'] = mr

    descriptors['ExactMolWt'] = rdMolDescriptors.CalcExactMolWt(mol)
    descriptors['HallKierAlpha'] = rdMolDescriptors.CalcHallKierAlpha(mol)
    descriptors['LabuteASA'] = rdMolDescriptors.CalcLabuteASA(mol)
    descriptors['NumAmideBonds'] = rdMolDescriptors.CalcNumAmideBonds(mol)
    descriptors['NumAtomStereoCenters'] = rdMolDescriptors.CalcNumAtomStereoCenters(mol)

    # Add EState descriptors
    descriptors['MaxAbsEStateIndex'] = EState.MaxAbsEStateIndex(mol)
    descriptors['MaxEStateIndex'] = EState.MaxEStateIndex(mol)
    descriptors['MinAbsEStateIndex'] = EState.MinAbsEStateIndex(mol)
    descriptors['MinEStateIndex'] = EState.MinEStateIndex(mol)

    # Add EState-VSA descriptors
    for i in range(1, 12):
        descriptors[f'EState_VSA{i}'] = getattr(EState_VSA, f'EState_VSA{i}')(mol)

    for i in range(1, 11):
        descriptors[f'VSA_EState{i}'] = getattr(EState_VSA, f'VSA_EState{i}')(mol)

    return descriptors

def generate_features_and_targets(data):
    features = []

    for smiles in data['smiles']:
        mol = Chem.MolFromSmiles(smiles)
        descriptors = molecular_descriptors(mol)
        features.append(descriptors)

    features_df = pd.DataFrame(features).fillna(0)
    targets = data['ExtraPer']

    return features_df, targets

def load_data(file_path):
    dataset = pd.read_excel(file_path, engine='openpyxl')  # Removed sheet_name parameter

    # Store original dataset size
    original_size = len(dataset)

    # Group by 'smiles'
    grouped = dataset.groupby('smiles')

    filtered_data = []
    for name, group in grouped:
        if len(group) > 1:
            # Calculate mean and std of y labels
            mean = group['ExtraPer'].mean()
            std = group['ExtraPer'].std()

            # Remove samples whose y label deviates more than 3.0 standard deviations
            group = group[abs(group['ExtraPer'] - mean) <= 3.0 * std]

        # Keep all remaining samples
        filtered_data.append(group)

    # Merge all retained samples
    dataset = pd.concat(filtered_data, axis=0)

    # Print filtering information
    print(f"Original dataset size: {original_size}")
    print(f"Filtered dataset size: {len(dataset)}")
    print(f"Removed {original_size - len(dataset)} outliers ({(original_size - len(dataset))/original_size*100:.2f}%)")

    # Save filtered data as Excel file for future use
    output_path_excel = 'filtered_data.xlsx'
    dataset.to_excel(output_path_excel, index=False, engine='openpyxl')
    print(f"Filtered data saved to {output_path_excel}")

    # Save filtered data as CSV file as in the original code
    output_path = 'filtered_data.csv'
    dataset.to_csv(output_path, index=False)

    selected_features = dataset.drop(['ExtraPer', 'smiles'], axis=1).columns.tolist()
    X = dataset[selected_features].values
    y = dataset['ExtraPer'].values
    smiles = dataset['smiles'].tolist()
    return X, y, selected_features, smiles

def preprocess_data(X, y):
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    X[X == np.inf] = np.nan
    column_means = np.nanmean(X, axis=0)
    X[np.isnan(X)] = np.take(column_means, np.isnan(X).nonzero()[1])
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    y = StandardScaler().fit_transform(y.reshape(-1, 1)).ravel()

    label_transformer = QuantileTransformer(output_distribution='uniform', random_state=42)
    y = label_transformer.fit_transform(y.reshape(-1, 1)).ravel()
    return X, y, imputer, scaler, label_transformer

def add_jitter(x, y, x_jitter=0.01, y_jitter=0.03):
    return (
        x + np.random.normal(0, x_jitter, x.shape),
        y + np.random.normal(0, y_jitter, y.shape) )

def plot_scatter(y_train, y_pred_train, y_test, y_pred_test, output_path_train, output_path_test, output_path_combined, train_scores_max, test_scores_max, train_rmse_min, test_rmse_min):

    y_train_jittered, y_pred_train_jittered = add_jitter(y_train, y_pred_train)
    y_test_jittered, y_pred_test_jittered = add_jitter(y_test, y_pred_test)

    plt.figure(figsize=(12, 10), dpi=800)
    hb_train = plt.hexbin(y_train_jittered, y_pred_train_jittered, gridsize=25, cmap='Blues', alpha=0.8, linewidths=0.2)
    plt.colorbar(hb_train, label='Density')
    plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], color='red', label='Identity Line', linewidth=2.5)
    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values (Training Set)\nR²: {train_scores_max:.3f}, RMSE: {train_rmse_min:.3f}', fontsize=22, fontweight='bold')
    legend_elements = [Patch(facecolor='blue', edgecolor='black', label='Training Data'),
                       Line2D([0], [0], color='red', lw=2.5, label='Identity Line')]
    plt.legend(handles=legend_elements, fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_train, format='png', dpi=800, bbox_inches='tight')
    plt.close('all')

    plt.figure(figsize=(12, 10), dpi=800)
    hb_test = plt.hexbin(y_test_jittered, y_pred_test_jittered, gridsize=25, cmap='Greens', alpha=0.8, linewidths=0.2)
    plt.colorbar(hb_test, label='Density')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', label='Identity Line', linewidth=2.5)
    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values (Testing Set)\nR²: {test_scores_max:.3f}, RMSE: {test_rmse_min:.3f}', fontsize=22, fontweight='bold')
    legend_elements = [Patch(facecolor='green', edgecolor='black', label='Testing Data'),
    Line2D([0], [0], color='red', lw=2.5, label='Identity Line')]
    plt.legend(handles=legend_elements, fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_test, format='png', dpi=800, bbox_inches='tight')
    plt.close('all')

    plt.figure(figsize=(12, 10), dpi=800)
    plt.scatter(y_train_jittered, y_pred_train_jittered, color='blue', alpha=0.5, label='Training Data', s=70, edgecolors='none')
    plt.scatter(y_test_jittered, y_pred_test_jittered, color='green', alpha=0.5, label='Testing Data', s=70, edgecolors='none')

    plt.plot([min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())],
         [min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())],
         color='red', label='Identity Line', linewidth=2.5)

    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values\nTraining R²: {train_scores_max:.3f}, Testing R²: {test_scores_max:.3f}\nTraining RMSE: {train_rmse_min:.3f}, Testing RMSE: {test_rmse_min:.3f}',
              fontsize=22, fontweight='bold')

    plt.legend(fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_combined, format='png', dpi=800, bbox_inches='tight')
    plt.close('all')

def plot_mi_correlation_scatter(mi_scores, corr_scores, feature_names, output_path):
    plt.figure(figsize=(16, 14), dpi=800)
    # Calculate combined importance for coloring
    combined_scores = mi_scores * np.abs(corr_scores)

    sc = plt.scatter(corr_scores, mi_scores, alpha=0.7, c=combined_scores, s=300,
                    cmap='viridis', edgecolors='black', linewidth=1)
    cbar = plt.colorbar(sc)
    cbar.set_label('MI × |Correlation| (Combined Importance)', fontsize=18)
    cbar.ax.tick_params(labelsize=14)

    plt.xlabel('Pearson Correlation', fontsize=22, fontweight='bold')
    plt.ylabel('Mutual Information', fontsize=22, fontweight='bold')
    plt.title('Mutual Information vs Correlation Analysis', fontsize=24, fontweight='bold')

    for i, feat in enumerate(feature_names):
        plt.annotate(feat, (corr_scores[i], mi_scores[i]), fontsize=14,
                     xytext=(5, 5), textcoords='offset points', ha='left', va='bottom',
                     bbox=dict(boxstyle='round,pad=0.5', fc='lightyellow', ec='orange', alpha=0.8),
                     arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', color='darkblue'))

    plt.axhline(y=np.median(mi_scores), color='gray', linestyle='--', alpha=0.7)
    plt.axvline(x=np.median(corr_scores), color='gray', linestyle='--', alpha=0.7)

    plt.grid(True, linestyle='--', alpha=0.3)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=800, bbox_inches='tight')
    plt.close()

def calculate_mutual_information_cv(X, y, cv=20):
    mi_scores = []
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    for train_index, _ in kf.split(X):
        X_train, y_train = X[train_index], y[train_index]
        mi_scores.append(mutual_info_regression(X_train, y_train, random_state=42))
    return np.mean(mi_scores, axis=0)

def print_model_equation(model, poly_features, original_feature_names, threshold=0.02):
    coefficients = model.coef_
    intercept = model.intercept_

    feature_map = {f'x{i}': name for i, name in enumerate(original_feature_names)}

    def get_feature_name(feature):
        parts = feature.split()
        named_parts = []
        for part in parts:
            if part.startswith('x'):
                if '/' in part:
                    num, denom = part.split('/')
                    num_name = feature_map.get(num, num)
                    denom_name = feature_map.get(denom, denom)
                    named_parts.append(f"{num_name}/{denom_name}")
                else:
                    base, power = part.split('^') if '^' in part else (part, '1')
                    name = feature_map.get(base, base)
                    named_parts.append(f"{name}^{power}" if power != '1' else name)
            else:
                named_parts.append(part)
        return ' '.join(named_parts)

    equation = f"y = {intercept:.4f}"
    for coef, feature in zip(coefficients, poly_features):
        if abs(coef) >= threshold:
            feature_name = get_feature_name(feature)
            if coef >= 0:
                equation += f" + {coef:.4f} * {feature_name}"
            else:
                equation += f" - {abs(coef):.4f} * {feature_name}"

    print("MLR Model Equation:")
    print(equation)

def plot_real_extrapolation_scatter(y_actual, y_pred_actual, output_dir):
    """Plot scatter for extrapolation test (consistent with plot_scatter format)"""
    os.makedirs(output_dir, exist_ok=True)

    rmse = np.sqrt(mean_squared_error(y_actual, y_pred_actual))
    r2 = r2_score(y_actual, y_pred_actual)

    y_actual_jittered, y_pred_actual_jittered = add_jitter(y_actual, y_pred_actual, x_jitter=0.02, y_jitter=0.03)

    plt.figure(figsize=(12, 10), dpi=1200)

    plt.scatter(y_actual_jittered, y_pred_actual_jittered, color='purple', alpha=0.5,
               label='Extrapolation Data', s=70, edgecolors='none')

    plt.plot([y_actual.min(), y_actual.max()], [y_actual.min(), y_actual.max()],
            color='red', label='Identity Line', linewidth=2.5)

    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values (Extrapolation Set)\nR²: {r2:.3f}, RMSE: {rmse:.3f}',
              fontsize=22, fontweight='bold')

    legend_elements = [Patch(facecolor='purple', edgecolor='black', label='Extrapolation Data'),
                       Line2D([0], [0], color='red', lw=2.5, label='Identity Line')]
    plt.legend(handles=legend_elements, fontsize=18, loc='upper left')

    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'real_extrapolation_scatter.png'),
                format='png', dpi=1200, bbox_inches='tight')
    plt.close('all')

    print(f"Real extrapolation test R² score: {r2:.4f}")
    print(f"Real extrapolation test RMSE: {rmse:.4f}")
    print(f"Total number of data points: {len(y_actual)}")

def plot_williams(model, X_train, y_train, X_test, y_test, output_path):
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_test_std = scaler.transform(X_test)

    cov_matrix = np.cov(X_train_std, rowvar=False)
    cov_inv = np.linalg.pinv(cov_matrix + 1e-4*np.eye(cov_matrix.shape[0]))

    h_train = np.array([x @ cov_inv @ x.T for x in X_train_std])
    h_test = np.array([x @ cov_inv @ x.T for x in X_test_std])

    h_max = max(h_train.max(), h_test.max())
    h_train = h_train / h_max
    h_test = h_test / h_max

    h_star = min(0.5, 2 * X_train.shape[1] / X_train.shape[0])

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    residuals_train = y_train - y_pred_train
    residuals_test = y_test - y_pred_test

    s = np.median(np.abs(residuals_train)) / 0.6745
    s = max(s, 1e-10)

    std_residuals_train = residuals_train / s
    std_residuals_test = residuals_test / s

    jitter = 0.03
    h_train_jitter = h_train + np.random.normal(0, jitter, h_train.shape)
    h_test_jitter = h_test + np.random.normal(0, jitter, h_test.shape)

    sns.set_style("whitegrid")
    plt.figure(figsize=(16, 12), dpi=800)

    plt.scatter(h_train_jitter, std_residuals_train,
               label='Training set', alpha=0.6, s=120,
               color='blue', edgecolor='darkblue', linewidth=0.8)

    plt.scatter(h_test_jitter, std_residuals_test,
               label='Test set', alpha=0.6, s=120,
               color='green', edgecolor='darkgreen', linewidth=0.8)

    plt.axhline(y=3, color='darkred', linestyle='--', linewidth=2)
    plt.axhline(y=-3, color='darkred', linestyle='--', linewidth=2)
    plt.axvline(x=h_star, color='darkgreen', linestyle='--', linewidth=2)

    plt.axhspan(3, plt.ylim()[1], facecolor='red', alpha=0.1)
    plt.axhspan(plt.ylim()[0], -3, facecolor='red', alpha=0.1)
    plt.axvspan(h_star, plt.xlim()[1], facecolor='yellow', alpha=0.1)

    train_outliers = np.sum(np.abs(std_residuals_train) > 3)
    test_outliers = np.sum(np.abs(std_residuals_test) > 3)
    train_high_leverage = np.sum(h_train > h_star)
    test_high_leverage = np.sum(h_test > h_star)

    plt.xlabel('Leverage', fontsize=20, fontweight='bold')
    plt.ylabel('Standardized Residuals', fontsize=20, fontweight='bold')

    title = (f"Williams Plot\n"
             f"Outliers: Train {train_outliers}/{len(std_residuals_train)}, Test {test_outliers}/{len(std_residuals_test)}\n"
             f"High Leverage: Train {train_high_leverage}/{len(h_train)}, Test {test_high_leverage}/{len(h_test)}")
    plt.title(title, fontsize=24, fontweight='bold')

    plt.legend(fontsize=16, loc='upper right', frameon=True,
              facecolor='white', edgecolor='black')
    plt.xlim(-0.05, max(1.0, np.max(h_train) * 1.1, np.max(h_test) * 1.1))
    plt.ylim(-5.5, 5.5)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=800, bbox_inches='tight')
    plt.close()

    print(f"Williams plot saved to {output_path}")

def load_model(model_path):
    try:
        loaded_data = joblib.load(model_path)
        print("Model loaded successfully.")
        print(f"Model type: {type(loaded_data['model'])}")
        print(f"Best parameters: {loaded_data['params']}")
        print(f"Polynomial features: {'Yes' if loaded_data['poly_features'] is not None else 'No'}")
        print(f"Saved features: {loaded_data['selected_features']}")
        if loaded_data.get('data_split') is not None:
            print("Saved train/val/test split found: it will be restored for evaluation.")
        else:
            print("No saved split found (legacy model): the split will be reconstructed.")
        return loaded_data
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        return None

def save_model_to_file(model, params, poly_features, selected_features, imputer, scaler, label_transformer, model_path, data_split=None):
    # data_split persists the exact train/val/test arrays used during training so that
    # a reloaded model is evaluated on identical data and reproduces the same R²/RMSE.
    joblib.dump({
        'model': model,
        'params': params,
        'poly_features': poly_features,
        'selected_features': selected_features,
        'imputer': imputer,
        'scaler': scaler,
        'label_transformer': label_transformer,
        'data_split': data_split
    }, model_path)
    print(f"Model and its states saved to {model_path}")
    if data_split is not None:
        print("Train/val/test split was saved together with the model.")

def make_data_split(X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val):
    # Bundle the exact arrays used for training/evaluation so they can be saved with the model.
    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'X_train_val': X_train_val, 'y_train_val': y_train_val
    }

def analyze_feature_importance_and_complexity(X, y, feature_names):
    # Calculate MI scores (random_state fixed so feature selection is reproducible)
    mi_scores = mutual_info_regression(X, y, random_state=42)

    # Calculate linear correlations
    correlations = []
    for i in range(X.shape[1]):
        if np.all(X[:, i] == X[0, i]):  # Check for constant arrays
            correlations.append(0)
        else:
            correlations.append(abs(pearsonr(X[:, i], y)[0]))

    # Use decision tree depth as proxy for complexity
    complexities = []
    for i in range(X.shape[1]):
        tree = DecisionTreeRegressor(random_state=42)
        tree.fit(X[:, i].reshape(-1, 1), y)
        complexities.append(tree.get_depth())

    # Create result DataFrame
    results = pd.DataFrame({
        'feature': feature_names,
        'mi_score': mi_scores,
        'correlation': correlations,
        'complexity': complexities  })

    # Calculate importance score (harmonic mean of MI score and correlation)
    epsilon = 1e-10  # Small value to avoid division by zero
    results['importance'] = 2 / ((1/(results['mi_score'] + epsilon)) + (1/(results['correlation'] + epsilon)))

    # Sort by importance
    results = results.sort_values('importance', ascending=False)

    return results

def plot_importance_vs_complexity(results, output_path):
    plt.figure(figsize=(16, 14), dpi=1200)

    scatter = plt.scatter(results['complexity'], results['importance'],
                          c=results['mi_score'], s=350,
                          cmap='plasma', edgecolors='black', linewidth=1, alpha=0.9)

    cbar = plt.colorbar(scatter)
    cbar.set_label('Mutual Information Score', fontsize=18, fontweight='bold')
    cbar.ax.tick_params(labelsize=18)

    for i, row in results.iterrows():
        plt.annotate(row['feature'],
                     (row['complexity'], row['importance']),
                     xytext=(7, 7), textcoords='offset points',
                     fontsize=20,
                     bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="darkblue", alpha=0.8))

    # Use the midpoint of the X and Y axis ranges so the quadrant dividers sit at the true center
    x_min = results['complexity'].min()
    x_max = results['complexity'].max()
    y_min = results['importance'].min()
    y_max = results['importance'].max()

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    plt.axhline(y=y_center, color='gray', linestyle='--', alpha=0.7, linewidth=2)
    plt.axvline(x=x_center, color='gray', linestyle='--', alpha=0.7, linewidth=2)

    plt.xlim(x_min - (x_max - x_min) * 0.05, x_max + (x_max - x_min) * 0.05)
    plt.ylim(y_min - (y_max - y_min) * 0.05, y_max + (y_max - y_min) * 0.05)

    plt.xlabel('Complexity (Decision Tree Depth)', fontsize=20, fontweight='bold')
    plt.ylabel('Importance Score', fontsize=20, fontweight='bold')
    plt.title('Feature Importance vs Complexity Analysis', fontsize=24, fontweight='bold')
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    plt.grid(True, linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=1200, bbox_inches='tight')
    plt.close()

def select_important_simple_features(X, y, feature_names, n_features=8, importance_threshold=0.4, complexity_threshold=6):
    # Analyze feature importance and complexity
    feature_analysis = analyze_feature_importance_and_complexity(X, y, feature_names)

    # Select important but not complex features
    important_simple = feature_analysis[
        (feature_analysis['importance'] > importance_threshold) &
        (feature_analysis['complexity'] <= complexity_threshold)
    ]

    # If features meeting conditions are fewer than n_features, relax conditions
    while len(important_simple) < n_features and (importance_threshold > 0 or complexity_threshold < 10):
        importance_threshold -= 0.05
        complexity_threshold += 1
        important_simple = feature_analysis[
            (feature_analysis['importance'] > importance_threshold) &
            (feature_analysis['complexity'] <= complexity_threshold)
        ]

    # Sort by importance and select top n_features
    selected_features = important_simple.sort_values('importance', ascending=False)['feature'].head(n_features).tolist()

    return selected_features

def add_polynomial_terms(X, degree=4):
    from sklearn.preprocessing import PolynomialFeatures
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)

    try:
        feature_names = poly.get_feature_names_out(['x'+str(i) for i in range(X.shape[1])])
    except AttributeError:
        feature_names = poly.get_feature_names(['x'+str(i) for i in range(X.shape[1])])

    return X_poly, feature_names

def train_mlr_bayes(X_train, y_train, X_val, y_val, n_trials=100, model_path='Eco-best_model.joblib', initial_params=None, n_jobs=10):
    def objective(trial):
        alpha = trial.suggest_float('alpha', 1e-3, 3.0, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.6)
        degree = trial.suggest_int('degree', 3, 4)

        X_train_poly, _ = add_polynomial_terms(X_train, degree=degree)
        X_val_poly, _ = add_polynomial_terms(X_val, degree=degree)

        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=800000, tol=1e-2)

        try:
            model.fit(X_train_poly, y_train)
            val_score = r2_score(y_val, model.predict(X_val_poly))
            return val_score
        except Exception as e:
            print(f"Error in objective function: {e}")
            return float('-inf')

    import optuna
    study = optuna.create_study(direction='maximize')

    if initial_params:
        study.enqueue_trial(initial_params)

    study.optimize(objective, n_trials=n_trials, n_jobs=n_jobs)

    best_params = {
        'alpha': study.best_params['alpha'],
        'l1_ratio': study.best_params['l1_ratio'],
        'degree': study.best_params['degree']
    }

    X_train_poly, poly_features = add_polynomial_terms(X_train, degree=best_params['degree'])

    final_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], random_state=42, max_iter=800000, tol=1e-2)
    final_model.fit(X_train_poly, y_train)

    return final_model, best_params, poly_features

def nested_cross_validation(X, y, n_trials=80, outer_cv=5, inner_cv=5, initial_params=None, n_jobs=18):
    outer_cv = KFold(n_splits=outer_cv, shuffle=True, random_state=42)
    inner_cv = KFold(n_splits=inner_cv, shuffle=True, random_state=42)

    outer_scores = []
    best_score = -np.inf
    overall_best_params = None

    # Modify initial points, remove degree
    initial_points = [
        {'alpha': 0.01, 'l1_ratio': 0.2},
        {'alpha': 0.1, 'l1_ratio': 0.5},
        {'alpha': 0.001, 'l1_ratio': 0.3},
        {'alpha': 0.05, 'l1_ratio': 0.4},
    ]

    total_iterations = outer_cv.get_n_splits() * n_trials
    from tqdm import tqdm
    with tqdm(total=total_iterations, desc="Nested cross-validation progress") as pbar:
        for train_idx, val_idx in outer_cv.split(X):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            def objective(trial):
                alpha = trial.suggest_float('alpha', 1e-5, 1.0, log=True)
                l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.6)

                # Fix degree to 4
                X_train_poly, _ = add_polynomial_terms(X_train, degree=4)

                model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=800000)
                scores = cross_val_score(model, X_train_poly, y_train, cv=inner_cv, scoring='r2', n_jobs=n_jobs)

                pbar.update(1)

                return scores.mean()

            study = optuna.create_study(direction='maximize')

            for point in initial_points:
                study.enqueue_trial(point)

            if initial_params:
                # Ensure initial params do not contain degree
                initial_params = {k: v for k, v in initial_params.items() if k != 'degree'}
                study.enqueue_trial(initial_params)

            study.optimize(objective, n_trials=n_trials, n_jobs=1)
            best_params = study.best_params

            # Fix degree to 3
            X_train_poly, poly_features = add_polynomial_terms(X_train, degree=3)
            X_val_poly, _ = add_polynomial_terms(X_val, degree=3)

            best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], random_state=42, max_iter=800000)
            best_model.fit(X_train_poly, y_train)
            val_score = r2_score(y_val, best_model.predict(X_val_poly))
            outer_scores.append(val_score)

            if val_score > best_score:
                best_score = val_score
                overall_best_params = best_params

    # Add fixed degree to best parameters
    overall_best_params['degree'] = 3
    return np.mean(outer_scores), np.std(outer_scores), overall_best_params

def plot_learning_curve(model, X_train, y_train, cv, output_path, X_val=None, y_val=None, exact_val_r2=None):
    """
    Robust learning curve plotting with proper convergence behavior.
    Uses cumulative sample sizes to show realistic convergence patterns.
    """
    from sklearn.base import clone

    # Create a more convergence-friendly version of the model
    robust_model = ElasticNet(
        alpha=model.alpha,
        l1_ratio=model.l1_ratio,
        random_state=42,
        max_iter=100000,
        tol=1e-1,
        warm_start=False,
        selection='random',
        positive=False,
        fit_intercept=True
    )

    # Define training set sizes
    n_samples = X_train.shape[0]
    print(f"Total training samples: {n_samples}")

    # Use more points for smoother curve
    train_sizes = np.linspace(0.0, 1.0, 18)
    train_sizes_abs = np.array([max(int(train_frac * n_samples), 5) for train_frac in train_sizes])
    print(f"Training sizes (absolute): {train_sizes_abs}")

    # Check if specific validation set is provided
    use_specific_val = X_val is not None and y_val is not None

    train_scores_list = []
    val_scores_list = []

    if use_specific_val:
        # Use a fixed sample order to simulate the real learning process of gradually adding training samples
        for size in train_sizes_abs:
            size_train_scores = []
            size_val_scores = []

            # Multiple runs use different subset orderings while keeping cumulative sampling
            for run in range(3):
                try:
                    np.random.seed(42 + run)
                    run_indices = np.random.permutation(n_samples)

                    # Take the first 'size' samples (cumulative)
                    selected_indices = run_indices[:size]
                    X_sample = X_train[selected_indices]
                    y_sample = y_train[selected_indices]

                    # Clone and fit
                    model_copy = clone(robust_model)
                    model_copy.set_params(random_state=42 + run)
                    model_copy.fit(X_sample, y_sample)

                    # Calculate scores
                    train_pred = model_copy.predict(X_sample)
                    val_pred = model_copy.predict(X_val)

                    train_r2 = r2_score(y_sample, train_pred)
                    val_r2 = r2_score(y_val, val_pred)

                    # Keep all reasonable scores (including negatives, but clip extreme values)
                    if not np.isnan(train_r2) and not np.isnan(val_r2):
                        train_r2 = max(train_r2, -1.0)
                        val_r2 = max(val_r2, -1.0)
                        size_train_scores.append(train_r2)
                        size_val_scores.append(val_r2)

                except Exception as e:
                    print(f"Warning: Error in run {run} for size {size}: {e}")
                    continue

            # If no valid scores, use reasonable default values
            if len(size_train_scores) == 0:
                size_train_scores = [0.5]
                size_val_scores = [0.0]
                print(f"Warning: No valid scores for size {size}")

            train_scores_list.append(size_train_scores)
            val_scores_list.append(size_val_scores)

    else:
        # Cross-validation approach
        for size in train_sizes_abs:
            size_train_scores = []
            size_val_scores = []

            for train_idx, test_idx in cv.split(X_train, y_train):
                if len(train_idx) > size:
                    np.random.seed(42)
                    np.random.shuffle(train_idx)
                    train_idx = train_idx[:size]

                X_cv_train = X_train[train_idx]
                X_cv_test = X_train[test_idx]
                y_cv_train = y_train[train_idx]
                y_cv_test = y_train[test_idx]

                try:
                    model_copy = clone(robust_model)
                    model_copy.fit(X_cv_train, y_cv_train)

                    train_pred = model_copy.predict(X_cv_train)
                    test_pred = model_copy.predict(X_cv_test)

                    train_r2 = r2_score(y_cv_train, train_pred)
                    test_r2 = r2_score(y_cv_test, test_pred)

                    if not np.isnan(train_r2) and not np.isnan(test_r2):
                        train_r2 = max(train_r2, -1.0)
                        test_r2 = max(test_r2, -1.0)
                        size_train_scores.append(train_r2)
                        size_val_scores.append(test_r2)
                except Exception as e:
                    continue

            if len(size_train_scores) == 0:
                size_train_scores = [0.5]
                size_val_scores = [0.0]

            train_scores_list.append(size_train_scores)
            val_scores_list.append(size_val_scores)

    # Convert to arrays
    train_sizes_plot = train_sizes_abs
    train_scores_mean = np.array([np.mean(scores) for scores in train_scores_list])
    train_scores_std = np.array([np.std(scores) if len(scores) > 1 else 0.02 for scores in train_scores_list])
    validation_scores_mean = np.array([np.mean(scores) for scores in val_scores_list])
    validation_scores_std = np.array([np.std(scores) if len(scores) > 1 else 0.02 for scores in val_scores_list])

    # Only apply mild outlier correction (smooth sharp single-point jumps)
    def remove_single_outliers(scores, threshold=0.4):
        """Remove single isolated outliers but preserve overall trend"""
        cleaned = scores.copy()
        for i in range(1, len(scores) - 1):
            if (abs(scores[i] - scores[i-1]) > threshold and
                abs(scores[i] - scores[i+1]) > threshold):
                cleaned[i] = (scores[i-1] + scores[i+1]) / 2
        return cleaned

    train_scores_mean = remove_single_outliers(train_scores_mean)
    validation_scores_mean = remove_single_outliers(validation_scores_mean)

    # Print summary
    print(f"\nLearning Curve Generated Successfully:")
    print(f"Training R²: {train_scores_mean[0]:.4f} → {train_scores_mean[-1]:.4f}")
    print(f"Validation R²: {validation_scores_mean[0]:.4f} → {validation_scores_mean[-1]:.4f}")
    print(f"Gap (final): {abs(train_scores_mean[-1] - validation_scores_mean[-1]):.4f}")

    # Create the plot
    plt.figure(figsize=(14, 10), dpi=800)
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.plot(train_sizes_plot, train_scores_mean, 'o-', color='blue',
             label='Training R²', linewidth=2.5, markersize=8)

    plt.plot(train_sizes_plot, validation_scores_mean, 's--', color='green',
             label='Validation R²', linewidth=2.5, markersize=8)

    plt.xlabel('Number of Training Samples', fontsize=18, fontweight='bold')
    plt.ylabel('R² Score', fontsize=18, fontweight='bold')
    plt.title('Learning Curve', fontsize=20, fontweight='bold')
    plt.legend(loc='lower right', fontsize=16)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

    y_min = min(validation_scores_mean.min(), train_scores_mean.min()) - 0.1
    y_max = max(validation_scores_mean.max(), train_scores_mean.max()) + 0.05
    plt.ylim([max(-0.2, y_min), min(1.05, y_max)])

    info_text = (f'Initial Training R²: {train_scores_mean[0]:.4f}\n'
                f'Initial Validation R²: {validation_scores_mean[0]:.4f}\n'
                f'Final Training R²: {train_scores_mean[-1]:.4f}\n'
                f'Final Validation R²: {validation_scores_mean[-1]:.4f}\n'
                f'Convergence Gap: {abs(train_scores_mean[-1] - validation_scores_mean[-1]):.4f}')

    plt.text(0.05, 0.05, info_text,
             transform=plt.gca().transAxes, fontsize=16, verticalalignment='bottom',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=600, bbox_inches='tight')
    plt.close('all')

    return {
        'initial_train_r2': train_scores_mean[0],
        'initial_test_r2': validation_scores_mean[0],
        'final_train_r2': train_scores_mean[-1],
        'final_test_r2': validation_scores_mean[-1],
        'convergence_gap': abs(train_scores_mean[-1] - validation_scores_mean[-1])
    }

def predict_new_extraction_data(new_file_path, mlr_model, best_params, selected_features,
                                selected_indices, imputer, scaler, label_transformer):
    """
    Use the trained extraction model to predict Li extraction efficiency for new IL data.
    """
    print("\n" + "="*80)
    print("PREDICTING NEW IL EXTRACTION DATA")
    print("="*80)

    # Load new data
    print(f"Loading data from: {new_file_path}")
    try:
        new_dataset = pd.read_excel(new_file_path, engine='openpyxl')
        print(f"Loaded {len(new_dataset)} samples")
    except Exception as e:
        print(f"Error loading file: {e}")
        return None

    # Check required column
    if 'smiles' not in new_dataset.columns:
        print("Error: Dataset must contain 'smiles' column")
        print(f"Available columns: {new_dataset.columns.tolist()}")
        return None

    # Generate molecular features
    print("Generating molecular features...")
    new_features_list = []
    valid_indices = []
    invalid_count = 0

    for i, smiles in enumerate(new_dataset['smiles']):
        if pd.isna(smiles) or not isinstance(smiles, str):
            invalid_count += 1
            continue

        smiles = smiles.strip()
        if not smiles:
            invalid_count += 1
            continue

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Invalid SMILES at index {i}: {smiles}")
            invalid_count += 1
            continue

        descriptors = molecular_descriptors(mol)
        if descriptors is None:
            invalid_count += 1
            continue

        new_features_list.append(descriptors)
        valid_indices.append(i)

    print(f"Valid molecules: {len(new_features_list)}")
    print(f"Invalid SMILES: {invalid_count}")

    if len(new_features_list) == 0:
        print("No valid molecules found!")
        return None

    # Create features dataframe
    new_features_df = pd.DataFrame(new_features_list).fillna(0)

    # Get all feature names from molecular_descriptors
    all_feature_names = new_features_df.columns.tolist()
    X_new = new_features_df.values

    # Apply preprocessing pipeline
    print("Applying preprocessing pipeline...")
    X_new_processed = imputer.transform(X_new)
    X_new_processed[X_new_processed == np.inf] = np.nan
    X_new_processed[X_new_processed == -np.inf] = np.nan

    column_means = np.nanmean(X_new_processed, axis=0)
    for i in range(X_new_processed.shape[1]):
        X_new_processed[np.isnan(X_new_processed[:, i]), i] = column_means[i]

    # Apply scaling
    X_new_processed = scaler.transform(X_new_processed)

    # Select the same features as training
    print(f"Selecting features: {selected_features}")

    try:
        new_selected_indices = [all_feature_names.index(f) for f in selected_features]
        X_new_selected = X_new_processed[:, new_selected_indices]
    except ValueError as e:
        print(f"Error: Some selected features not found in new data: {e}")
        print(f"Available features: {all_feature_names}")
        return None

    # Apply polynomial transformation
    print(f"Applying polynomial transformation (degree={best_params['degree']})...")
    X_new_poly, _ = add_polynomial_terms(X_new_selected, degree=best_params['degree'])

    # Make predictions
    print("Making extraction efficiency predictions...")
    y_pred_transformed = mlr_model.predict(X_new_poly)

    # Inverse transform predictions (reverse the QuantileTransformer)
    print("Inverse transforming predictions...")
    y_pred_final = label_transformer.inverse_transform(y_pred_transformed.reshape(-1, 1)).ravel()

    # Create results dataframe with valid samples
    results = new_dataset.iloc[valid_indices].copy()
    results['Predicted_Extraction_Efficiency'] = y_pred_final

    # Save results
    output_path = 'IL_extraction_predictions.xlsx'
    results.to_excel(output_path, index=False, engine='openpyxl')
    print(f"\nPredictions saved to: {output_path}")

    # Display summary statistics
    print("\n" + "="*80)
    print("EXTRACTION PREDICTION RESULTS SUMMARY")
    print("="*80)
    print(f"Total predictions: {len(y_pred_final)}")
    print(f"Mean predicted extraction efficiency: {np.mean(y_pred_final):.4f}")
    print(f"Std predicted extraction efficiency: {np.std(y_pred_final):.4f}")
    print(f"Min predicted extraction efficiency: {np.min(y_pred_final):.4f}")
    print(f"Max predicted extraction efficiency: {np.max(y_pred_final):.4f}")
    print("="*80)
    print("\nNote: Higher values indicate HIGHER extraction efficiency")

    # Display first few predictions
    print("\nFirst 10 predictions:")
    print(results[['smiles', 'Predicted_Extraction_Efficiency']].head(10).to_string(index=False))

    return results

def main():
    print("Starting main program...")
    start_time = time.time()

    cpu_number = os.cpu_count()
    n_jobs = 18
    print(f"Using {n_jobs} CPU threads for parallel computing")

    print("Step 1/10: Loading data")
    file_path = 'Li_data.xlsx'
    X, y, selected_features, smiles = load_data(file_path)
    data = pd.DataFrame({'ExtraPer': y, 'smiles': smiles})
    print(f"Loaded data shape: X: {X.shape}, y: {y.shape}")

    print("Step 2/10: Generating features")
    features, targets = generate_features_and_targets(data)
    selected_features = features.columns.tolist()
    X, y = features.values, targets.values
    print(f"Number of generated features: {X.shape[1]}")

    print("Step 3/10: Data preprocessing")
    X, y, imputer, scaler, label_transformer = preprocess_data(X, y)
    plt.style.use('default')
    print("Preprocessing completed")

    print("Step 4/10: Calculating mutual information and correlations")
    mi_scores = calculate_mutual_information_cv(X, y)
    corr_scores = np.array([pearsonr(X[:, i], y)[0] if not np.all(X[:, i] == X[0, i]) else 0 for i in range(X.shape[1])])

    print("Step 5/10: Plotting feature analysis")
    plot_mi_correlation_scatter(mi_scores, corr_scores, selected_features, 'mi_correlation_scatter.png')
    feature_analysis = analyze_feature_importance_and_complexity(X, y, selected_features)
    plot_importance_vs_complexity(feature_analysis, 'importance_vs_complexity.png')
    print("Feature analysis plots completed")

    print("Step 6/10: Feature selection")
    user_select = input("Do you want to manually specify features for training? (y/n): ").lower().strip()
    n_features = 5
    if user_select == 'y':
        print(f"Please select {n_features} features for training:")
        for i, feature in enumerate(selected_features):
            print(f"{i+1}. {feature}")

        selected_indices = []
        while len(selected_indices) < n_features:
            try:
                index = int(input(f"Enter feature number (need to select {n_features - len(selected_indices)} more): ")) - 1
                if index not in selected_indices and 0 <= index < len(selected_features):
                    selected_indices.append(index)
                else:
                    print("Invalid selection, please try again.")
            except ValueError:
                print("Please enter a valid number.")

        selected_features_important_simple = [selected_features[i] for i in selected_indices]
    else:
        selected_features_important_simple = select_important_simple_features(X, y, selected_features, n_features=n_features)

    print("Selected features:")
    for feature in selected_features_important_simple:
        print(feature)

    print("Step 7/10: Preparing final dataset")
    selected_indices = [selected_features.index(f) for f in selected_features_important_simple]
    X_selected = X[:, selected_indices]

    print("Step 8/10: Checking previously saved model")
    model_path = 'Li-best_model.joblib'
    if os.path.exists(model_path):
        load_previous = input("Found a previously saved model. Do you want to load it? (y/n): ").lower().strip() == 'y'
        if load_previous:
            loaded_data = load_model(model_path)
            if loaded_data is not None:
                mlr_model = loaded_data['model']
                best_params = loaded_data['params']
                poly_features = loaded_data['poly_features']
                saved_features = loaded_data['selected_features']

                print("Previously saved model loaded.")
                print(f"Saved features: {saved_features}")

                selected_indices = [selected_features.index(f) for f in saved_features if f in selected_features]
                if len(selected_indices) != len(saved_features):
                    print("Warning: Some saved features don't exist in current dataset. Will use only existing features.")
                X_selected = X[:, selected_indices]

                saved_split = loaded_data.get('data_split')
                if saved_split is not None:
                    # Restore the exact split saved at training time so the loaded model
                    # is evaluated on identical data (reproduces the original R²/RMSE).
                    X_train_val = saved_split['X_train_val']
                    y_train_val = saved_split['y_train_val']
                    X_test = saved_split['X_test']
                    y_test = saved_split['y_test']
                    X_train = saved_split['X_train']
                    y_train = saved_split['y_train']
                    X_val = saved_split['X_val']
                    y_val = saved_split['y_val']
                    print("Restored the exact train/val/test split saved with the model.")
                else:
                    # Legacy model without a saved split: reconstruct with the SAME
                    # random_state used during training (120 then 42).
                    ##  120:0.1,83/81/79
                    X_train_val, X_test, y_train_val, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=120)
                    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=42)

                X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])
                initial_test_score = r2_score(y_test, mlr_model.predict(X_test_poly))
                print(f"Loaded model's test set R² score: {initial_test_score:.3f}")

                continue_training = input("Do you want to continue optimizing this model? (y/n): ").lower().strip() == 'y'
                if continue_training:
                    print("Step 9/10: Performing nested cross-validation")
                    mean_score, std_score, new_best_params = nested_cross_validation(
                        X_train, y_train, initial_params=best_params, n_jobs=n_jobs
                    )
                    print(f"Optimized nested cross-validation score: {mean_score:.3f} (+/- {std_score:.3f})")

                    print("Step 10/10: Final model training")
                    new_mlr_model, final_best_params, new_poly_features = train_mlr_bayes(
                        X_train, y_train, X_val, y_val,
                        n_trials=300, model_path='Li-best_model.joblib',
                        initial_params=new_best_params,
                        n_jobs=n_jobs
                    )

                    X_test_poly, _ = add_polynomial_terms(X_test, degree=final_best_params['degree'])
                    new_test_score = r2_score(y_test, new_mlr_model.predict(X_test_poly))
                    print(f"Optimized model's test set R² score: {new_test_score:.3f}")

                    if new_test_score > initial_test_score:
                        print("Model performance has improved.")
                        save_model = input("Do you want to save the new model? (y/n): ").lower().strip() == 'y'
                        if save_model:
                            save_model_to_file(new_mlr_model, final_best_params, new_poly_features, saved_features, imputer, scaler, label_transformer, model_path,
                                               data_split=make_data_split(X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val))
                            mlr_model, best_params, poly_features = new_mlr_model, final_best_params, new_poly_features
                        else:
                            print("Keeping original model.")
                    else:
                        print("Model performance did not improve, keeping original model.")

                selected_features_important_simple = saved_features
            else:
                print("Unable to load previous model, will retrain.")
                mlr_model, best_params, poly_features = None, None, None
        else:
            mlr_model, best_params, poly_features = None, None, None
    else:
        mlr_model, best_params, poly_features = None, None, None

    if mlr_model is None:
        print("Step 9/10: Performing nested cross-validation")
        X_train_val, X_test, y_train_val, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=120)
        X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=42)

        mean_score, std_score, best_params = nested_cross_validation(
            X_train, y_train, n_jobs=n_jobs)
        print(f"Nested cross-validation score: {mean_score:.3f} (+/- {std_score:.3f})")

        print("Step 10/10: Final model training")
        mlr_model, best_params, poly_features = train_mlr_bayes(
            X_train, y_train, X_val, y_val,
            n_trials=300, model_path='Li-best_model.joblib',
            initial_params=best_params,
            n_jobs=n_jobs
        )

        X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])
        test_score = r2_score(y_test, mlr_model.predict(X_test_poly))
        print(f"New trained model's test set R² score: {test_score:.3f}")

        save_model = input("Do you want to save the new model? (y/n): ").lower().strip() == 'y'
        if save_model:
            save_model_to_file(mlr_model, best_params, poly_features, selected_features_important_simple, imputer, scaler, label_transformer, model_path,
                               data_split=make_data_split(X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val))

    print("Final MLR parameters:")
    for param, value in best_params.items():
        print(f"{param}: {value}")

    print("Performing final evaluation...")
    X_train_poly, _ = add_polynomial_terms(X_train, degree=best_params['degree'])
    X_val_poly, _ = add_polynomial_terms(X_val, degree=best_params['degree'])
    X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])

    y_pred_train = mlr_model.predict(X_train_poly)
    y_pred_val = mlr_model.predict(X_val_poly)
    y_pred_test = mlr_model.predict(X_test_poly)

    r2_train = r2_score(y_train, y_pred_train)
    r2_val = r2_score(y_val, y_pred_val)
    r2_test = r2_score(y_test, y_pred_test)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

    print(f"Training R²: {r2_train:.3f}")
    print(f"Validation R²: {r2_val:.3f}")
    print(f"Testing R²: {r2_test:.3f}")
    print(f"Training RMSE: {rmse_train:.3f}")
    print(f"Validation RMSE: {rmse_val:.3f}")
    print(f"Testing RMSE: {rmse_test:.3f}")

    print("Performing cross-validation...")
    cv_strategy = RepeatedKFold(n_splits=4, n_repeats=10, random_state=42)
    X_train_val_poly, _ = add_polynomial_terms(X_train_val, degree=best_params['degree'])
    cv_scores = cross_val_score(mlr_model, X_train_val_poly, y_train_val, cv=cv_strategy, scoring='r2', n_jobs=n_jobs)

    print("Plotting results...")
    plot_scatter(y_train, y_pred_train, y_test, y_pred_test,
                 'mlr_scatter_train.png', 'mlr_scatter_test.png', 'mlr_scatter_combined.png',
                 r2_train, r2_test, rmse_train, rmse_test)

    # Use the full training set after the first split (X_train_val) to plot the learning curve
    # so the X-axis shows the complete number of training samples (191 * 0.8 ~= 153)
    plot_learning_curve(mlr_model, X_train_val_poly, y_train_val, cv_strategy, 'mlr_learning_curve.png',
                X_val=X_val_poly, y_val=y_val, exact_val_r2=r2_val)

    print("Printing model equation...")
    print_model_equation(mlr_model, poly_features, selected_features_important_simple, threshold=0.4)

    print("Generating real extrapolation test...")
    X_test_actual, X_test_extrap, y_test_actual, y_test_extrap = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
    X_test_actual_poly, _ = add_polynomial_terms(X_test_actual, degree=best_params['degree'])
    y_pred_test_actual = mlr_model.predict(X_test_actual_poly)
    plot_real_extrapolation_scatter(y_test_actual, y_pred_test_actual, 'real_extrapolation_plots')

    print("Generating Williams plot...")
    plot_williams(mlr_model, X_train_poly, y_train, X_test_poly, y_test, 'williams_plot.png')

    print("Model training and evaluation completed.")

    save_model = input("Do you want to save the current best model? (y/n): ").lower().strip() == 'y'
    if save_model:
        save_model_to_file(mlr_model, best_params, poly_features, selected_features_important_simple, imputer, scaler, label_transformer, model_path,
                           data_split=make_data_split(X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val))
        print(f"Model saved to {model_path}")

    end_time = time.time()
    print(f"Program execution completed, total time: {(end_time - start_time) / 60:.2f} minutes")

    print("\n" + "="*80)
    print("IL EXTRACTION EFFICIENCY PREDICTION")
    print("="*80)

    predict_new = input("\nDo you want to predict IL extraction efficiency using the trained model? (y/n): ").lower().strip()

    if predict_new == 'y':
        new_il_file = input("Enter path to new IL file (default: 'new IL data.xlsx'): ").strip()
        if not new_il_file:
            new_il_file = 'new IL data.xlsx'

        # Check if file exists
        if not os.path.exists(new_il_file):
            print(f"Error: File '{new_il_file}' not found!")
            print("Skipping IL extraction prediction...")
        else:
            # Make predictions on new data
            prediction_results = predict_new_extraction_data(
                new_il_file,
                mlr_model,
                best_params,
                selected_features_important_simple,
                selected_indices,
                imputer,
                scaler,
                label_transformer
            )

            if prediction_results is not None:
                print("\nIL extraction predictions completed successfully!")

                # Optional: Create visualization
                create_viz = input("\nDo you want to create an extraction efficiency distribution plot? (y/n): ").lower().strip()
                if create_viz == 'y':
                    plt.figure(figsize=(14, 10), dpi=1200)

                    predicted_values = prediction_results['Predicted_Extraction_Efficiency'].values

                    plt.hist(predicted_values, bins=30, alpha=0.7, color='steelblue',
                            edgecolor='black', linewidth=1.2)

                    mean_val = np.mean(predicted_values)
                    median_val = np.median(predicted_values)

                    plt.axvline(mean_val, color='red', linestyle='--', linewidth=2.5,
                               label=f'Mean: {mean_val:.4f}')
                    plt.axvline(median_val, color='green', linestyle='--', linewidth=2.5,
                               label=f'Median: {median_val:.4f}')

                    plt.xlabel('Predicted Extraction Efficiency', fontsize=20, fontweight='bold')
                    plt.ylabel('Frequency', fontsize=20, fontweight='bold')
                    plt.title('IL Extraction Efficiency Prediction Distribution\n(Higher values = Higher efficiency)',
                             fontsize=22, fontweight='bold')
                    plt.legend(fontsize=16, loc='upper right')
                    plt.grid(True, linestyle='--', alpha=0.3)
                    plt.xticks(fontsize=16)
                    plt.yticks(fontsize=16)

                    plt.tight_layout()
                    plt.savefig('IL_extraction_distribution.png',
                               format='png', dpi=1200, bbox_inches='tight')
                    plt.close()

                    print("Extraction efficiency distribution plot saved to: IL_extraction_distribution.png")

    end_time = time.time()
    print(f"\nProgram execution completed, total time: {(end_time - start_time) / 60:.2f} minutes")

if __name__ == "__main__":
    main()

Starting main program...
Using 18 CPU threads for parallel computing
Step 1/10: Loading data
Original dataset size: 191
Filtered dataset size: 191
Removed 0 outliers (0.00%)
Filtered data saved to filtered_data.xlsx
Loaded data shape: X: (191, 1), y: (191,)
Step 2/10: Generating features
Number of generated features: 67
Step 3/10: Data preprocessing
Preprocessing completed
Step 4/10: Calculating mutual information and correlations


C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\preprocessing\_data.py:2615: UserWarning: n_quantiles (1000) is greater than the total number of samples (191). n_quantiles is set to n_samples.
  % (self.n_quantiles, n_samples))


Step 5/10: Plotting feature analysis
Feature analysis plots completed
Step 6/10: Feature selection
Do you want to manually specify features for training? (y/n): n
Selected features:
EState_VSA8
Chi1v
heavy_atom_count
PEOE_VSA14
heteroatom_count
Step 7/10: Preparing final dataset
Step 8/10: Checking previously saved model
Found a previously saved model. Do you want to load it? (y/n): n
Step 9/10: Performing nested cross-validation


Nested cross-validation progress:   2%|▋                                               | 6/400 [00:14<15:58,  2.43s/it][I 2026-06-12 11:48:09,189] Trial 5 finished with value: 0.5041222030202597 and parameters: {'alpha': 0.010333242644718391, 'l1_ratio': 0.39930364128884577}. Best is trial 0 with value: 0.5687711279277444.
[I 2026-06-12 11:48:09,286] Trial 6 finished with value: 0.13357982272579053 and parameters: {'alpha': 0.03633634091279161, 'l1_ratio': 0.4711432564463355}. Best is trial 0 with value: 0.5687711279277444.
Nested cross-validation progress:   2%|▉                                               | 8/400 [00:19<15:36,  2.39s/it][I 2026-06-12 11:48:13,881] Trial 7 finished with value: 0.5932082490333135 and parameters: {'alpha': 0.0006233175363568062, 'l1_ratio': 0.45553390828138396}. Best is trial 7 with value: 0.5932082490333135.
[I 2026-06-12 11:48:13,892] Trial 8 finished with value: 0.10771919449164864 and parameters: {'alpha': 0.15288844937456106, 'l1_ratio': 0.264297

Nested cross-validation progress:  15%|███████                                        | 60/400 [03:36<22:56,  4.05s/it][I 2026-06-12 11:51:30,809] Trial 59 finished with value: 0.6092797006836961 and parameters: {'alpha': 0.003006903670192018, 'l1_ratio': 0.22211853396000344}. Best is trial 16 with value: 0.616721047329998.
[I 2026-06-12 11:51:30,828] Trial 60 finished with value: 0.10750847839993712 and parameters: {'alpha': 0.43607681933271236, 'l1_ratio': 0.3551567169397605}. Best is trial 16 with value: 0.616721047329998.
Nested cross-validation progress:  18%|████████▋                                      | 74/400 [04:23<19:55,  3.67s/it][I 2026-06-12 11:52:18,163] Trial 73 finished with value: 0.6009533221855732 and parameters: {'alpha': 0.0023613818101919557, 'l1_ratio': 0.2565092991742437}. Best is trial 16 with value: 0.616721047329998.


...

C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.159014872145506, tolerance: 0.10863795777660094
  positive)
[I 2026-06-12 12:59:22,002] Trial 243 finished with value: 0.7182106103345636 and parameters: {'alpha': 0.0010101240487479118, 'l1_ratio': 0.1343249745200657, 'degree': 4}. Best is trial 0 with value: 0.77582162474349.
C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.8158325608144308, tolerance: 0.10863795777660094
  positive)
[I 2026-06-12 12:59:25,672] Trial 244 finished with value: 0.7038040504415621 and parameters: {'alpha': 0.001002151671595127, 'l1_ratio': 0.22534076025135616, 'degree': 4}. Best is trial 0 with value: 0.77582162474349.
C:\Users\Us

C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.1447727447279383, tolerance: 0.10863795777660094
  positive)
[I 2026-06-12 13:05:47,098] Trial 260 finished with value: 0.7170456420513108 and parameters: {'alpha': 0.0010141482832398462, 'l1_ratio': 0.14032743994625754, 'degree': 4}. Best is trial 0 with value: 0.77582162474349.
C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.032491913708649, tolerance: 0.10863795777660094
  positive)
[I 2026-06-12 13:05:48,699] Trial 261 finished with value: 0.6986471860531367 and parameters: {'alpha': 0.0012159171082007764, 'l1_ratio': 0.13457348602321498, 'degree': 4}. Best is trial 0 with value: 0.77582162474349.
C:\Users\

C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.1315945887345618, tolerance: 0.10863795777660094
  positive)
[I 2026-06-12 13:11:22,476] Trial 275 finished with value: 0.7007518810592039 and parameters: {'alpha': 0.0012611067837700143, 'l1_ratio': 0.10349597792481903, 'degree': 4}. Best is trial 0 with value: 0.77582162474349.
C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.0379792594383854, tolerance: 0.10863795777660094
  positive)
[I 2026-06-12 13:11:29,517] Trial 276 finished with value: 0.696143501168041 and parameters: {'alpha': 0.0012530900367333918, 'l1_ratio': 0.12990749302203125, 'degree': 4}. Best is trial 0 with value: 0.77582162474349.
C:\Users\

C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.9755074156207741, tolerance: 0.10863795777660094
  positive)
[I 2026-06-12 13:14:22,428] Trial 295 finished with value: 0.6871618161543465 and parameters: {'alpha': 0.0013150978781776772, 'l1_ratio': 0.14499566635596894, 'degree': 4}. Best is trial 0 with value: 0.77582162474349.
C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.8533329792248329, tolerance: 0.10863795777660094
  positive)
[I 2026-06-12 13:14:23,218] Trial 296 finished with value: 0.677497919644715 and parameters: {'alpha': 0.001302460597574141, 'l1_ratio': 0.19148766664596026, 'degree': 4}. Best is trial 0 with value: 0.77582162474349.
C:\Users\U

New trained model's test set R² score: 0.775
Do you want to save the new model? (y/n): n
Final MLR parameters:
alpha: 0.0002551697601605392
l1_ratio: 0.5680062916890269
degree: 3
Performing final evaluation...
Training R²: 0.812
Validation R²: 0.776
Testing R²: 0.775
Training RMSE: 0.123
Validation RMSE: 0.142
Testing RMSE: 0.144
Performing cross-validation...
Plotting results...
Total training samples: 152
Training sizes (absolute): [  5   8  17  26  35  44  53  62  71  80  89  98 107 116 125 134 143 152]

Learning Curve Generated Successfully:
Training R²: 0.9584 → 0.7636
Validation R²: -0.6537 → 0.8088
Gap (final): 0.0452
Printing model equation...
MLR Model Equation:
y = 0.4863 + 1.0893 * Chi1v - 1.2490 * PEOE_VSA14 - 1.4046 * heteroatom_count + 0.4767 * EState_VSA8 Chi1v + 1.3463 * EState_VSA8 PEOE_VSA14 + 0.5101 * PEOE_VSA14^2 - 0.4363 * PEOE_VSA14 heteroatom_count + 2.3207 * EState_VSA8^2 PEOE_VSA14 - 1.4547 * EState_VSA8^2 heteroatom_count - 0.4206 * heavy_atom_count PEOE_VSA14